# 05 — Explainability (XAI)

Two complementary explainability modes for Chapter 6 — one *intrinsic* (free with the architecture), one *post-hoc* (general-purpose). Per Blueprint §10 and the test-set verdict from notebook 04: **the cross-attention architecture's headline contribution is intrinsic explainability**, not F1 — concat_fusion is actually slightly better on test F1 (+0.42 pt). What hemt_clip uniquely offers is the attention heatmaps generated here.

**Method 1 — Cross-attention heatmaps (intrinsic, on `hemt_clip`):**
- Per Blueprint §10.1: extract `(B, num_heads, 1, 196)` attention weights from `CrossAttentionFusion`, average over heads, reshape to a **14×14 patch grid** (ViT-B/16 at 224 px), bilinear-upsample to 224×224, overlay with the original image. Tells us *where on the image the text-conditioned model was looking* for evidence.
- Sample selection (default 3 per bucket = 12 total): `{correct, wrong}` × `{high-confidence, low-confidence}`. Correct + high-conf shows the heatmap "working as intended"; wrong + high-conf is the most informative bucket for error analysis (where was the model confidently looking when it got it wrong?).

**Method 2 — SHAP on text (post-hoc, on `text_only`):**
- Per Blueprint §10.2: `shap.Explainer` with `shap.maskers.Text` over the text_only model. Attributes each title's prediction to its individual tokens. We use *text_only* (not hemt_clip's text branch) for a clean methodological question: SHAP attributes a model's predictions, so the cleanest answer to "which words pushed the verdict?" comes from a model whose verdict is a function of text alone.
- Sample selection (default 30): stratified across `{correct, wrong}` × `{real, fake}` with three confidence quantiles per cell.
- Multimodal SHAP intentionally skipped per Blueprint §10.2 — marginal value for FYP scope, considerable complexity.

**Inputs:** `outputs/eval/preds_hemt_clip.npz` + `outputs/eval/preds_text_only.npz` from `notebooks/04_evaluation.ipynb`, plus the canonical `*_best.pt` checkpoints.

**Outputs (under `outputs/xai/`):** attention overlays (12 individual + 1 composite grid), SHAP per-sample bars (30) + aggregate top-tokens figure + per-token CSV.

**Runtime: ~8–12 min on T4** (attention is fast; SHAP is the cost — ~5–10 min for 30 text-only forward passes per token mask).

## Setup
Same idempotent bootstrap as notebooks 02–04. Mount Drive (Account B in the OAuth popup), repo pull, deps install, `jax`/`flax` removal, HDF5 → local SSD.

In [1]:
# Bootstrap — idempotent. Safe to re-run on a fresh or warm Colab runtime.
import os, sys, subprocess, shutil

os.environ['USE_FLAX'] = 'FALSE'
os.environ['USE_TF'] = 'FALSE'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
REPO_URL = 'https://github.com/staharizvi/hemt-clip-fnd.git'
REPO_DIR = '/content/hemt-clip-fnd'
H5_DRIVE = '/content/drive/MyDrive/hemt-clip-fnd/data/fakeddit.h5'
H5_LOCAL = '/content/fakeddit.h5'

if IN_COLAB:
    if not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print('Drive already mounted.')

    if os.path.exists(os.path.join(REPO_DIR, '.git')):
        print('Repo present — pulling latest…')
        subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'], check=True)
    else:
        print('Cloning repo…')
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)

    subprocess.run(['pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'], check=True)
    subprocess.run(['pip', 'uninstall', '-y', '-q', 'jax', 'jaxlib', 'flax'], check=False)

    if not os.path.exists(H5_LOCAL):
        if os.path.exists(H5_DRIVE):
            print(f'Copying {H5_DRIVE} -> {H5_LOCAL}…')
            shutil.copy(H5_DRIVE, H5_LOCAL)
        else:
            print(f'WARNING: {H5_DRIVE} not found.')
    else:
        print(f'h5 already at {H5_LOCAL}.')

    os.chdir(REPO_DIR)

print('\ncwd:', os.getcwd())
print('h5 :', H5_LOCAL, 'exists:', os.path.exists(H5_LOCAL))

Drive already mounted.
Repo present — pulling latest…
h5 already at /content/fakeddit.h5.

cwd: /content/hemt-clip-fnd
h5 : /content/fakeddit.h5 exists: True


## Point scripts at the local HDF5
Same yaml patch as nb 03/04 — local SSD reads beat Drive FUSE.

In [2]:
import yaml, pathlib

cfg_path = pathlib.Path('configs/base.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
if cfg['data']['hdf5_path'] != '/content/fakeddit.h5':
    cfg['data']['hdf5_path'] = '/content/fakeddit.h5'
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print('Patched cfg.data.hdf5_path -> /content/fakeddit.h5')
else:
    print('cfg already points at local HDF5.')
print('checkpoints :', cfg['checkpointing']['dir'])

cfg already points at local HDF5.
checkpoints : /content/drive/MyDrive/hemt-clip-fnd/checkpoints


## Discover the two checkpoints we need

Same auto-discovery logic as notebook 04 (`training.evaluate.discover_checkpoints`): prefer non-`_seed*` files per variant, then latest mtime. We need:

- `hemt_clip` → v4 seed=42 ckpt (the canonical headline run; cross-attention weights are the source of the heatmaps).
- `text_only` → latest text_only ckpt (text branch is backbone-independent so the B/16 era run is equivalent to the v2 one).

In [3]:
from pathlib import Path

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from training.evaluate import discover_checkpoints

ckpt_dir = Path(cfg['checkpointing']['dir'])
ckpts = discover_checkpoints(ckpt_dir)
HEMT_CLIP_CKPT = ckpts['hemt_clip']
TEXT_ONLY_CKPT = ckpts['text_only']
print('hemt_clip  ckpt:', HEMT_CLIP_CKPT.name)
print('text_only  ckpt:', TEXT_ONLY_CKPT.name)

# Confirm preds_*.npz are present (produced by notebook 04)
preds_hemt = Path('outputs/eval/preds_hemt_clip.npz')
preds_text = Path('outputs/eval/preds_text_only.npz')
for p in (preds_hemt, preds_text):
    print(p, 'exists:', p.exists())

hemt_clip  ckpt: hemt_hemt_clip_20260530-0223_best.pt
text_only  ckpt: hemt_text_only_20260530-0103_best.pt
outputs/eval/preds_hemt_clip.npz exists: False
outputs/eval/preds_text_only.npz exists: False


### Auto-recover missing preds files

The attention and SHAP scripts both consume `outputs/eval/preds_{hemt_clip,text_only}.npz` from notebook 04. Those files live under `/content/hemt-clip-fnd/outputs/eval/` — which is **not persisted** between Colab runtimes. If you're on a fresh runtime, the next cell will rerun `training.evaluate` to regenerate them (~3 min on T4). If they're already present, the cell is a no-op.

This makes notebook 05 self-sufficient — you can run it standalone without bouncing into notebook 04 first.

In [ ]:
# Ensure the preds_*.npz files exist. They're produced by `training.evaluate`
# (notebook 04) — but Colab wipes /content on every fresh runtime, so if you didn't
# just re-run notebook 04 in this session, they're gone. Auto-recover by running
# evaluate.py now (~3 min on T4) rather than asking you to bounce notebooks.
import subprocess

needed = [Path('outputs/eval/preds_hemt_clip.npz'),
          Path('outputs/eval/preds_text_only.npz')]
missing = [p for p in needed if not p.exists()]

if missing:
    print(f'{len(missing)}/{len(needed)} preds_*.npz missing — running training.evaluate (~3 min on T4)...')
    print('(also regenerates the eval figures + summary table; idempotent.)\n')
    rc = subprocess.run([sys.executable, '-m', 'training.evaluate', '--split', 'test']).returncode
    if rc != 0:
        raise RuntimeError(f'training.evaluate failed with rc={rc}; fix before continuing.')
    for p in needed:
        assert p.exists(), f'{p} still missing after evaluate.py finished — investigate.'
    print('\nDone — preds files now present.')
else:
    print('All preds_*.npz already present — skipping eval.')

## Method 1 — Cross-attention heatmaps (intrinsic, `hemt_clip`)

`explainability.attention_viz` picks 12 test examples stratified by `{correct, wrong}` × `{high-conf, low-conf}` (default 3 per bucket), runs each through `hemt_clip` to capture per-head attention weights from `CrossAttentionFusion`, averages over the 8 heads, reshapes to 14×14, bilinear-upsamples to 224×224, and overlays on the original image.

**Expected wall-clock: ~30s on T4** (12 single-sample forward passes).

**What to look for in the report:**
- `correct_hi`: heatmap should localize on semantically relevant image regions (faces, objects mentioned in the title).
- `wrong_hi`: the most informative bucket — where was the model confidently looking when it got the answer wrong? Often reveals image artefacts the model latched onto, or text that's misaligned with the image.
- `correct_lo` / `wrong_lo`: borderline calls; useful for showing the heatmap's behaviour on uncertain cases.

In [4]:
!python -m explainability.attention_viz \
    --checkpoint "{HEMT_CLIP_CKPT}" \
    --preds-npz outputs/eval/preds_hemt_clip.npz \
    --n-per-bucket 3

2026-05-30 15:06:02,850 [INFO] device=cuda
2026-05-30 15:06:02,850 [INFO] loading preds from outputs/eval/preds_hemt_clip.npz
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/hemt-clip-fnd/explainability/attention_viz.py", line 285, in <module>
    sys.exit(main())
             ^^^^^^
  File "/content/hemt-clip-fnd/explainability/attention_viz.py", line 215, in main
    npz = np.load(args.preds_npz)
          ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/npyio.py", line 427, in load
    fid = stack.enter_context(open(os_fspath(file), "rb"))
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'outputs/eval/preds_hemt_clip.npz'


In [5]:
from IPython.display import Image, display, Markdown
import json

attn_dir = Path('outputs/xai/attention')
manifest = json.loads((attn_dir / 'attention_manifest.json').read_text(encoding='utf-8'))

display(Markdown('### Composite grid — all 12 examples'))
display(Image(filename=str(attn_dir / 'attention_grid.png')))

display(Markdown('### Per-example panels (text snippets included)'))
for entry in manifest:
    display(Markdown(
        f"**[{entry['bucket']}]** pred={'fake' if entry['pred']==1 else 'real'} "
        f"(conf={entry['confidence']:.3f}) | true={'fake' if entry['label']==1 else 'real'}"
    ))
    display(Image(filename=str(attn_dir / entry['file'])))

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/xai/attention/attention_manifest.json'

## Method 2 — SHAP for text (post-hoc, `text_only`)

`explainability.shap_text` runs `shap.Explainer` with a `shap.maskers.Text` masker over the **text_only variant**. Attributes each title's prediction to its individual tokens via Owen-value partitioning (faster than KernelExplainer, equivalent semantics).

Why text_only, not hemt_clip's text branch: SHAP attributes a model's predictions. We want the cleanest possible answer to "which words pushed the verdict?" — and that's the model whose verdict is exactly a function of text alone. Mixing in image features would muddy the attribution.

**Expected wall-clock: ~5–10 min on T4** (30 samples × ~10–20 model calls per sample for the partition tree).

**Outputs:**
- 30 individual per-sample bar plots in `outputs/xai/shap/shap_NN_*.png`
- `shap_top_tokens.png` — aggregate top-15 tokens by mean SHAP value toward each class
- `shap_token_records.csv` — long-form per-token SHAP values across all samples

In [ ]:
!python -m explainability.shap_text \
    --checkpoint "{TEXT_ONLY_CKPT}" \
    --preds-npz outputs/eval/preds_text_only.npz \
    --n-samples 30

In [ ]:
shap_dir = Path('outputs/xai/shap')

display(Markdown('### Aggregate — top tokens pushing toward each class'))
display(Image(filename=str(shap_dir / 'shap_top_tokens.png')))

shap_manifest = json.loads((shap_dir / 'shap_manifest.json').read_text(encoding='utf-8'))

# Show only the wrong + high-confidence ones for Chapter 6 — the most informative.
wrong_hi = sorted(
    [m for m in shap_manifest if m['status'] == 'wrong'],
    key=lambda m: -m['confidence'],
)[:6]
display(Markdown(f"### Confident errors (top 6 of {sum(1 for m in shap_manifest if m['status']=='wrong')} wrong predictions)"))
for entry in wrong_hi:
    display(Markdown(
        f"**Sample {entry['ds_idx']}** | pred={'fake' if entry['pred']==1 else 'real'} "
        f"(conf={entry['confidence']:.3f}) | true={'fake' if entry['label']==1 else 'real'}"
    ))
    display(Image(filename=str(shap_dir / entry['file'])))

# A couple of confident successes for contrast.
correct_hi = sorted(
    [m for m in shap_manifest if m['status'] == 'correct'],
    key=lambda m: -m['confidence'],
)[:4]
display(Markdown(f"### Confident successes (top 4)"))
for entry in correct_hi:
    display(Markdown(
        f"**Sample {entry['ds_idx']}** | pred={'fake' if entry['pred']==1 else 'real'} "
        f"(conf={entry['confidence']:.3f}) | true={'fake' if entry['label']==1 else 'real'}"
    ))
    display(Image(filename=str(shap_dir / entry['file'])))

## Take-aways for Chapter 6 (fill in after the run)

Structure first; plug in observations once the figures are concrete.

**Attention heatmaps (intrinsic, hemt_clip):**
- **Correct + high-conf**: does the heatmap localize on faces / objects / salient regions consistent with what the title is talking about? If yes — clean qualitative evidence that the cross-attention is doing semantically meaningful work, which is the report's headline contribution.
- **Wrong + high-conf**: what is the model attending to when it's confidently wrong? Common patterns to look for: heatmap on image artefacts (watermarks, text overlays), heatmap on the *background* rather than the subject, heatmap diffuse / unfocused (no clear localization).
- **Reproducibility**: same 12 samples each run (`--seed 42`). The composite grid figure (`attention_grid.png`) is the single chapter-ready figure; per-example PNGs are for the appendix or oral defence.

**SHAP on text (post-hoc, text_only):**
- **Aggregate top tokens**: do the FAKE-leaning tokens make narrative sense (sensational words, modifiers, satire markers)? Do the REAL-leaning tokens look like neutral news vocabulary (organizations, dates, geographic markers)?
- **Confident errors**: when text_only gets a high-conf wrong answer, which tokens did SHAP attribute? Common pattern on Fakeddit: real-sounding titles with manipulated images (text_only has no way to know), or genuinely satirical/ambiguous titles whose label is debatable.
- **Confident successes**: do positive token contributions match human intuition? Sanity-check that the model isn't latching onto spurious features (e.g., every title starting with `breaking:` getting flagged fake).

**Joint framing for Chapter 6.4 (Qualitative Analysis):**
- Two complementary modalities of explanation: intrinsic (image, via cross-attention) + post-hoc (text, via SHAP). This pairing — *intrinsic on one modality, post-hoc on the other* — is the report's contribution to the "unified two-method explainability framework" claim from the existing report's Novel Contributions section.
- The intrinsic vs post-hoc distinction is itself a defensible methodological point in viva (intrinsic ≈ no perturbation cost, no model approximation, but architecture-specific; post-hoc ≈ general-purpose, perturbation-based, but more compute and approximation error). Worth one paragraph in 6.4.

**Next:** Chapter 6 outline + draft, using `outputs/eval/summary_test.{csv,md}` as the results table, the figures from notebook 04 for the quantitative section, and these XAI figures for §6.4 (Qualitative Analysis). Notebook 06 (Streamlit demo) is then the final engineering artefact before report writing.